[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/A.%20Foundational%20Analytics%20%26%20Inventory%20Control%20%28The%20Language%20of%20Supply%20Chain%29/052.%20The%20EOQ%20with%20Quantity%20Discounts%20Problem/P52-Tier-2_executed.ipynb)

---


# 52. The EOQ with Quantity Discounts Problem

## Tier 2: The Three-Step EOQ Procedure

### Purpose and scope

This notebook applies the classic three-step procedure to an economic order quantity (EOQ) problem with all-units quantity discounts. The input is annual demand, the fixed cost of placing an order, an annual holding-rate fraction, and a list of quantity ranges with their unit prices. The output is a candidate order quantity and annual cost for each tier, followed by the lowest-cost candidate.

The procedure follows the standard EOQ model and evaluates each discount tier separately. It is often called a heuristic in practical presentations; under the assumptions below and a continuous order-quantity model, minimizing the cost within every tier and then comparing those minima gives the global minimum over the tier ranges being evaluated.

### Model assumptions

- Annual demand is known, constant, and fully satisfied; shortages are not allowed.
- Each order incurs the same fixed ordering cost.
- The annual holding cost per unit is the holding-rate fraction multiplied by that tier's unit price.
- The listed prices are all-units prices: once a tier is selected, its unit price applies to all units demanded during the year.
- Replenishment is modeled using the standard EOQ assumptions. Lead time, safety stock, capacity limits, taxes, and other operational constraints are outside this model.
- The EOQ calculation treats order quantity as continuous. Tier limits are evaluated as the numeric bounds supplied in the data; in applications requiring strictly integer quantities, the candidate quantities should be interpreted as EOQ approximations.

### Intuition: why evaluate every tier?

A lower unit price reduces annual purchase cost, but qualifying for that price may require a larger order. Larger orders increase average inventory and therefore holding cost. For each tier, the procedure first finds the EOQ implied by that tier's price. If that EOQ is outside the tier's quantity range, the nearest feasible boundary is the least-cost quantity within that tier. The procedure then compares the resulting annual costs across tiers.

### Mathematical model

For annual demand \(D>0\), fixed ordering cost \(S>0\), holding-rate fraction \(h>0\), unit price \(c_i>0\) in tier \(i\), and order quantity \(Q>0\), the annual total cost within tier \(i\) is

$$
TC_i(Q) = Dc_i + \frac{DS}{Q} + \frac{h c_i Q}{2}.
$$

The terms are, respectively, annual purchase cost, annual ordering cost, and annual holding cost. The factor \(Q/2\) represents average cycle inventory when inventory decreases steadily from \(Q\) to zero between replenishments.

Without tier limits, differentiating the variable-cost terms with respect to \(Q\) gives the standard EOQ:

$$
Q_i^* = \sqrt{\frac{2DS}{h c_i}}.
$$

For a tier with lower bound \(L_i\) and upper bound \(U_i\), the procedure evaluates

$$
\widehat{Q}_i = \begin{cases}
L_i, & Q_i^* < L_i, \\
U_i, & Q_i^* > U_i \text{ and } U_i < \infty, \\
Q_i^*, & \text{otherwise},
\end{cases}
\qquad
\widehat{TC}_i = TC_i(\widehat{Q}_i),
\qquad
\text{select a tier minimizing } \widehat{TC}_i.
$$

Here, \(Q_i^*\) is the unconstrained EOQ at tier \(i\)'s price; \(\widehat{Q}_i\) is that tier's feasible candidate; and \(\widehat{TC}_i\) is its annual cost. The cost function is convex in \(Q>0\), so its minimum over a tier's interval is the EOQ when feasible, or the closest interval boundary otherwise. Comparing these per-tier minima gives the minimum over the tier intervals considered. The code implements these equations in `calculate_eoq`, `calculate_total_cost`, and `solve`.

For \(n\) tiers, the procedure evaluates each tier once, so its time complexity is \(O(n)\). The implementation stores one result per tier, requiring \(O(n)\) space. Equal-cost candidates are treated as ties using a numerical tolerance; the first tied tier in the supplied list is returned as the representative solution, and all tied tiers are identified in the detailed results.

### Three-step procedure

1. **Calculate an EOQ for each tier.** The unit price determines the holding cost per unit and therefore the EOQ for that tier.
2. **Make the EOQ feasible for that tier.** If it is outside the tier's range, use the nearest range boundary.
3. **Compare annual total costs.** Select the candidate with the lowest cost, while reporting any numerically tied tiers.

### Concrete example

The two-tier example uses these all-units prices:

| Tier | Order-quantity range (units) | Unit price |
|---|---:|---:|
| 1 | 0–999 | $5.00 |
| 2 | 1,000 and above | $4.75 |

The annual demand is \(D=2400\) units, the ordering cost is \(S=\$50\) per order, and the holding-rate fraction is \(h=0.20\). The code calculates and displays the EOQ, feasible candidate, and annual cost for each tier rather than assuming that the lowest unit price is automatically the least-cost choice.

### Reading the outputs and visualizations

The detailed solution output shows how each candidate is obtained and why an EOQ is adjusted, if an adjustment is needed. The first figure visualizes candidate quantities, cost comparisons, the three-step sequence, and measured execution time as the number of tiers varies. The timing panel measures this procedure itself; it does not compare it with a different optimization method. The extended example applies the same procedure to four tiers, then visualizes its cost components, tier-specific candidates, and the effect of changing demand.

The sensitivity analysis changes demand only for the extended example. It illustrates how the selected tier and candidate quantity respond to that parameter; it does not change the base example's data.

### Strengths and limitations

The procedure is transparent, requires no optimization solver, and scales linearly with the number of tiers. Its conclusions depend on the EOQ assumptions and the tier ranges supplied. It does not model coupled decisions across multiple products or additional constraints such as storage capacity. When using discrete order quantities or more detailed operational constraints, the model assumptions and candidate quantities must reflect those requirements.

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
class EOQQuantityDiscount:
    """Apply the three-step EOQ procedure to all supplied discount tiers."""

    def __init__(self, demand, ordering_cost, holding_rate, discount_tiers):
        """Store model inputs and check that they are usable.

        Args:
            demand: Annual demand in units per year.
            ordering_cost: Fixed cost per order.
            holding_rate: Annual holding-cost fraction of unit price.
            discount_tiers: Tuples of (minimum quantity, maximum quantity, unit price).
        """
        if demand <= 0:
            raise ValueError("Annual demand must be positive.")
        if ordering_cost <= 0:
            raise ValueError("Ordering cost must be positive.")
        if holding_rate <= 0:
            raise ValueError("Holding rate must be positive.")
        if not discount_tiers:
            raise ValueError("At least one discount tier is required.")

        for min_qty, max_qty, unit_cost in discount_tiers:
            if min_qty < 0 or max_qty < min_qty or unit_cost <= 0:
                raise ValueError("Each tier must have valid bounds and a positive unit price.")

        self.demand = demand
        self.ordering_cost = ordering_cost
        self.holding_rate = holding_rate
        self.discount_tiers = discount_tiers

    def calculate_eoq(self, unit_cost):
        """Calculate the unconstrained EOQ for a tier's unit price."""
        return math.sqrt(
            (2 * self.demand * self.ordering_cost)
            / (self.holding_rate * unit_cost)
        )

    def calculate_total_cost(self, quantity, unit_cost):
        """Calculate annual holding, ordering, and purchase costs."""
        if quantity <= 0:
            raise ValueError("Order quantity must be positive.")

        holding_cost = (quantity * unit_cost * self.holding_rate) / 2
        ordering_cost = (self.demand * self.ordering_cost) / quantity
        purchase_cost = self.demand * unit_cost
        return holding_cost + ordering_cost + purchase_cost

    def solve(self, detailed_output=False):
        """Evaluate one EOQ-based candidate per tier and select the least-cost one.

        Returns:
            With detailed_output=True, return the selected result and all tier results.
            Otherwise, return (selected quantity, annual cost, selected tier).
        """
        results = []

        # Each tier has its own holding cost, so its EOQ must use that tier's price.
        for i, (min_qty, max_qty, unit_cost) in enumerate(self.discount_tiers):
            eoq = self.calculate_eoq(unit_cost)

            if eoq < min_qty:
                adjusted_qty = min_qty
                adjustment_reason = "Below minimum; use the minimum quantity"
            elif max_qty != float("inf") and eoq > max_qty:
                adjusted_qty = max_qty
                adjustment_reason = "Above maximum; use the maximum quantity"
            else:
                adjusted_qty = eoq
                adjustment_reason = "Within range; no adjustment"

            total_cost = self.calculate_total_cost(adjusted_qty, unit_cost)
            results.append(
                {
                    "tier": i + 1,
                    "min_qty": min_qty,
                    "max_qty": max_qty,
                    "unit_cost": unit_cost,
                    "eoq": eoq,
                    "adjusted_qty": adjusted_qty,
                    "total_cost": total_cost,
                    "adjustment_reason": adjustment_reason,
                    "feasible_eoq": min_qty <= eoq <= max_qty,
                }
            )

        minimum_cost = min(result["total_cost"] for result in results)
        tied_results = [
            result
            for result in results
            if math.isclose(
                result["total_cost"],
                minimum_cost,
                rel_tol=1e-12,
                abs_tol=1e-9,
            )
        ]
        for result in results:
            result["is_optimal"] = result in tied_results

        # A stable list order makes the representative result deterministic in a tie.
        optimal_result = tied_results[0]
        optimal_result["tied_optimal_tiers"] = [
            result["tier"] for result in tied_results
        ]

        if detailed_output:
            return optimal_result, results

        return (
            optimal_result["adjusted_qty"],
            optimal_result["total_cost"],
            optimal_result["tier"],
        )


# Base example: annual demand, ordering cost, and holding-rate fraction.
demand = 2400
ordering_cost = 50
holding_rate = 0.20

discount_tiers = [
    (0, 999, 5.00),
    (1000, float("inf"), 4.75),
]

print("Problem parameters:")
print(f"Annual demand: {demand} units/year")
print(f"Ordering cost: ${ordering_cost:.2f} per order")
print(f"Holding rate: {holding_rate * 100:.0f}% of unit price per year")
print("\nDiscount tiers:")
for i, (min_qty, max_qty, unit_cost) in enumerate(discount_tiers):
    max_str = f"{max_qty:.0f}" if max_qty != float("inf") else "and above"
    print(f"Tier {i + 1}: {min_qty}–{max_str} units at ${unit_cost:.2f} each")

In [ ]:
# Apply the three steps and retain each tier's intermediate result.
solver = EOQQuantityDiscount(demand, ordering_cost, holding_rate, discount_tiers)
optimal_result, all_results = solver.solve(detailed_output=True)

print("=" * 72)
print("THREE-STEP EOQ PROCEDURE")
print("=" * 72)

print("\nSteps 1 and 2: Calculate the EOQ and make it feasible for each tier")
for result in all_results:
    max_str = (
        f"{result['max_qty']:.0f}"
        if result["max_qty"] != float("inf")
        else "and above"
    )
    print(f"\nTier {result['tier']}:")
    print(f"  Unit price: ${result['unit_cost']:.2f}")
    print(f"  Quantity range: {result['min_qty']}–{max_str}")
    print(f"  Calculated EOQ: {result['eoq']:.2f} units")
    print(f"  EOQ within tier range: {result['feasible_eoq']}")
    print(f"  Candidate quantity: {result['adjusted_qty']:.2f} units")
    print(f"  Candidate choice: {result['adjustment_reason']}")
    print(f"  Annual total cost: ${result['total_cost']:,.2f}")

print("\nStep 3: Select a minimum-cost candidate")
print(f"  Selected quantity: {optimal_result['adjusted_qty']:.2f} units")
print(f"  Minimum annual cost: ${optimal_result['total_cost']:,.2f}")
print(f"  Representative tier: {optimal_result['tier']}")
print(f"  Unit price: ${optimal_result['unit_cost']:.2f}")
if len(optimal_result["tied_optimal_tiers"]) > 1:
    print(f"  Tied minimum-cost tiers: {optimal_result['tied_optimal_tiers']}")

# Check that the selected candidate has minimum cost and satisfies its tier bounds.
assert math.isclose(
    optimal_result["total_cost"],
    min(result["total_cost"] for result in all_results),
    rel_tol=1e-12,
    abs_tol=1e-9,
)
assert (
    optimal_result["min_qty"]
    <= optimal_result["adjusted_qty"]
    <= optimal_result["max_qty"]
)
assert math.isclose(
    optimal_result["total_cost"],
    solver.calculate_total_cost(
        optimal_result["adjusted_qty"], optimal_result["unit_cost"]
    ),
    rel_tol=1e-12,
    abs_tol=1e-9,
)
print("\nValidation checks passed.")

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle(
    "Three-Step EOQ Procedure with Quantity Discounts",
    fontsize=16,
    fontweight="bold",
)

# Panels 1 and 2 display the quantities and costs calculated for each tier.
step_df = pd.DataFrame(
    [
        {
            "Tier": f"Tier {result['tier']}",
            "EOQ": result["eoq"],
            "Candidate quantity": result["adjusted_qty"],
            "Annual cost": result["total_cost"],
            "Selected": result["is_optimal"],
        }
        for result in all_results
    ]
)
x_pos = np.arange(len(step_df))

ax1.bar(
    x_pos - 0.2,
    step_df["EOQ"],
    0.4,
    label="Calculated EOQ",
    alpha=0.8,
    color="lightblue",
)
ax1.bar(
    x_pos + 0.2,
    step_df["Candidate quantity"],
    0.4,
    label="Feasible candidate",
    alpha=0.8,
    color="lightcoral",
)
ax1.set_xlabel("Discount tier")
ax1.set_ylabel("Order quantity (units)")
ax1.set_title("Steps 1 and 2: EOQ and tier adjustment")
ax1.set_xticks(x_pos)
ax1.set_xticklabels(step_df["Tier"])
ax1.legend()
ax1.grid(True, alpha=0.3)

for i, result in enumerate(all_results):
    if not math.isclose(result["adjusted_qty"], result["eoq"]):
        ax1.annotate(
            "Adjusted",
            xy=(i, result["adjusted_qty"]),
            xytext=(i, max(step_df["Candidate quantity"]) * 1.1),
            ha="center",
            fontsize=8,
            color="darkred",
            arrowprops={"arrowstyle": "->", "color": "darkred", "alpha": 0.7},
        )

colors = ["seagreen" if selected else "skyblue" for selected in step_df["Selected"]]
ax2.bar(step_df["Tier"], step_df["Annual cost"], color=colors, alpha=0.85)
ax2.set_xlabel("Discount tier")
ax2.set_ylabel("Annual total cost ($)")
ax2.set_title("Step 3: Compare candidate costs")
ax2.grid(True, alpha=0.3)
for i, result in enumerate(all_results):
    if result["is_optimal"]:
        ax2.annotate(
            f"Minimum\n${result['total_cost']:,.0f}",
            xy=(i, result["total_cost"]),
            xytext=(i, result["total_cost"] * 1.05),
            ha="center",
            fontsize=9,
            fontweight="bold",
            color="darkgreen",
            arrowprops={"arrowstyle": "->", "color": "darkgreen"},
        )

# The flow diagram makes the repeated per-tier calculation and final comparison visible.
flow_steps = [
    "Calculate EOQ\nfor each tier",
    "Check tier range\nand adjust",
    "Calculate annual\ntotal costs",
    "Select the least-cost\ncandidate",
]
flow_x = [0.13, 0.38, 0.63, 0.88]
for i, (step, x) in enumerate(zip(flow_steps, flow_x)):
    ax3.text(
        x,
        0.5,
        step,
        ha="center",
        va="center",
        bbox={"boxstyle": "round,pad=0.4", "facecolor": "lightblue", "alpha": 0.8},
        fontsize=9,
        fontweight="bold",
    )
    if i < len(flow_steps) - 1:
        ax3.annotate(
            "",
            xy=(flow_x[i + 1] - 0.09, 0.5),
            xytext=(x + 0.09, 0.5),
            arrowprops={"arrowstyle": "->", "lw": 2, "color": "blue"},
        )
ax3.set_xlim(0, 1)
ax3.set_ylim(0, 1)
ax3.set_title("Three-step procedure")
ax3.axis("off")

# Measure this procedure as tier count grows; the generated tiers are a deterministic
# benchmark workload, not additional input to the base example.
problem_sizes = [2, 3, 5, 10, 15, 20]
benchmark_repeats = 100
average_times = []

for size in problem_sizes:
    tiers = []
    base_cost = 10.0
    for i in range(size):
        min_qty = i * 500
        max_qty = (i + 1) * 500 - 1 if i < size - 1 else float("inf")
        unit_cost = base_cost * (0.95**i)
        tiers.append((min_qty, max_qty, unit_cost))

    benchmark_solver = EOQQuantityDiscount(
        demand, ordering_cost, holding_rate, tiers
    )
    start_time = time.perf_counter()
    for _ in range(benchmark_repeats):
        benchmark_solver.solve()
    elapsed = time.perf_counter() - start_time
    average_times.append(elapsed / benchmark_repeats)

ax4.plot(
    problem_sizes,
    average_times,
    "o-",
    label="Three-step procedure",
    linewidth=2,
    markersize=6,
)
ax4.set_xlabel("Number of discount tiers")
ax4.set_ylabel("Average execution time per solve (seconds)")
ax4.set_title(f"Measured execution time ({benchmark_repeats} runs per size)")
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Apply the same three-step procedure to a four-tier example.
print("=" * 72)
print("EXTENDED EXAMPLE: FOUR DISCOUNT TIERS")
print("=" * 72)

extended_tiers = [
    (0, 499, 22.00),
    (500, 1249, 19.50),
    (1250, 2999, 18.25),
    (3000, float("inf"), 17.00),
]
extended_demand = 6500
extended_ordering_cost = 95
extended_holding_rate = 0.22

print(f"\nAnnual demand: {extended_demand} units/year")
print(f"Ordering cost: ${extended_ordering_cost:.2f} per order")
print(f"Holding rate: {extended_holding_rate * 100:.0f}% of unit price per year")
print("\nDiscount tiers:")
for i, (min_qty, max_qty, unit_cost) in enumerate(extended_tiers):
    max_str = f"{max_qty:.0f}" if max_qty != float("inf") else "and above"
    print(f"Tier {i + 1}: {min_qty}–{max_str} units at ${unit_cost:.2f} each")

extended_solver = EOQQuantityDiscount(
    extended_demand,
    extended_ordering_cost,
    extended_holding_rate,
    extended_tiers,
)
extended_optimal, extended_results = extended_solver.solve(detailed_output=True)

print("\nPer-tier candidates:")
for result in extended_results:
    print(
        f"Tier {result['tier']}: EOQ={result['eoq']:.2f}, "
        f"candidate quantity={result['adjusted_qty']:.2f}, "
        f"annual cost=${result['total_cost']:,.2f}"
    )

print("\nSelected solution:")
print(f"Order quantity: {extended_optimal['adjusted_qty']:.2f} units")
print(f"Annual total cost: ${extended_optimal['total_cost']:,.2f}")
print(f"Selected tier: {extended_optimal['tier']}")
print(f"Unit price: ${extended_optimal['unit_cost']:.2f}")
if len(extended_optimal["tied_optimal_tiers"]) > 1:
    print(f"Tied minimum-cost tiers: {extended_optimal['tied_optimal_tiers']}")

In [ ]:
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Extended Analysis: Four-Tier Discount Structure", fontsize=16, fontweight="bold")

# Plot each tier's cost only over its own quantity range.
quantities = np.linspace(100, 5000, 200)
for i, (min_qty, max_qty, unit_cost) in enumerate(extended_tiers):
    upper_limit = max_qty if max_qty != float("inf") else quantities[-1]
    costs = [
        extended_solver.calculate_total_cost(quantity, unit_cost)
        if min_qty <= quantity <= upper_limit
        else np.nan
        for quantity in quantities
    ]
    ax1.plot(
        quantities,
        costs,
        label=f"Tier {i + 1}: ${unit_cost:.2f} per unit",
        linewidth=2,
    )

ax1.plot(
    extended_optimal["adjusted_qty"],
    extended_optimal["total_cost"],
    "r*",
    markersize=15,
    label=f"Selected: {extended_optimal['adjusted_qty']:.0f} units",
)
ax1.set_xlabel("Order quantity (units)")
ax1.set_ylabel("Annual total cost ($)")
ax1.set_title("Annual cost within each tier")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Show the three annual cost components beside their total for each candidate.
tier_comparison = []
for result in extended_results:
    quantity = result["adjusted_qty"]
    unit_cost = result["unit_cost"]
    holding = quantity * unit_cost * extended_holding_rate / 2
    ordering = extended_demand * extended_ordering_cost / quantity
    purchase = extended_demand * unit_cost
    tier_comparison.append(
        {
            "Tier": f"Tier {result['tier']}",
            "Holding": holding,
            "Ordering": ordering,
            "Purchase": purchase,
            "Total": result["total_cost"],
        }
    )

tier_df = pd.DataFrame(tier_comparison)
x = np.arange(len(tier_df))
width = 0.2
ax2.bar(x - 1.5 * width, tier_df["Holding"], width, label="Holding cost")
ax2.bar(x - 0.5 * width, tier_df["Ordering"], width, label="Ordering cost")
ax2.bar(x + 0.5 * width, tier_df["Purchase"], width, label="Purchase cost")
ax2.bar(x + 1.5 * width, tier_df["Total"], width, label="Total cost", color="red")
ax2.set_xlabel("Discount tier")
ax2.set_ylabel("Annual cost ($)")
ax2.set_title("Cost components at each tier's candidate")
ax2.set_xticks(x)
ax2.set_xticklabels(tier_df["Tier"])
ax2.legend()
ax2.grid(True, alpha=0.3)

# Comparing EOQs with feasible candidates reveals which tier bounds are active.
eoq_vals = [result["eoq"] for result in extended_results]
adjusted_vals = [result["adjusted_qty"] for result in extended_results]
tier_nums = [result["tier"] for result in extended_results]
ax3.plot(tier_nums, eoq_vals, "o-", label="Calculated EOQ", linewidth=2, markersize=8)
ax3.plot(
    tier_nums,
    adjusted_vals,
    "s-",
    label="Feasible candidate",
    linewidth=2,
    markersize=8,
)
ax3.set_xlabel("Discount tier")
ax3.set_ylabel("Order quantity (units)")
ax3.set_title("EOQ and tier-feasible candidate")
ax3.legend()
ax3.grid(True, alpha=0.3)

# Demand sensitivity keeps the tier structure, ordering cost, and holding rate fixed.
demand_variations = np.linspace(0.5, 2.0, 16)
optimal_tiers = []
optimal_quantities = []
for variation in demand_variations:
    varied_solver = EOQQuantityDiscount(
        extended_demand * variation,
        extended_ordering_cost,
        extended_holding_rate,
        extended_tiers,
    )
    quantity, _, tier = varied_solver.solve()
    optimal_tiers.append(tier)
    optimal_quantities.append(quantity)

ax4_twin = ax4.twinx()
ax4.plot(
    demand_variations * 100,
    optimal_quantities,
    "b-",
    linewidth=2,
    marker="o",
    markersize=4,
    label="Selected quantity",
)
ax4_twin.plot(
    demand_variations * 100,
    optimal_tiers,
    "r--",
    linewidth=2,
    marker="s",
    markersize=4,
    label="Selected tier",
)
ax4.set_xlabel("Demand (% of extended-example demand)")
ax4.set_ylabel("Selected quantity (units)", color="blue")
ax4_twin.set_ylabel("Selected tier", color="red")
ax4.set_title("Sensitivity to annual demand")
ax4.grid(True, alpha=0.3)
lines, labels = ax4.get_legend_handles_labels()
twin_lines, twin_labels = ax4_twin.get_legend_handles_labels()
ax4.legend(lines + twin_lines, labels + twin_labels, loc="best")

plt.tight_layout()
plt.show()

In [ ]:
print("=" * 72)
print("THREE-STEP EOQ PROCEDURE: SUMMARY")
print("=" * 72)

print("\nBase example:")
print(f"  Selected quantity: {optimal_result['adjusted_qty']:.2f} units")
print(f"  Minimum annual cost: ${optimal_result['total_cost']:,.2f}")
print(f"  Representative tier: {optimal_result['tier']}")
print(f"  Unit price: ${optimal_result['unit_cost']:.2f}")
if len(optimal_result["tied_optimal_tiers"]) > 1:
    print(f"  Tied minimum-cost tiers: {optimal_result['tied_optimal_tiers']}")

print("\nFour-tier example:")
print(f"  Selected quantity: {extended_optimal['adjusted_qty']:.2f} units")
print(f"  Minimum annual cost: ${extended_optimal['total_cost']:,.2f}")
print(f"  Representative tier: {extended_optimal['tier']}")
print(f"  Unit price: ${extended_optimal['unit_cost']:.2f}")
if len(extended_optimal["tied_optimal_tiers"]) > 1:
    print(f"  Tied minimum-cost tiers: {extended_optimal['tied_optimal_tiers']}")

print("\nAlgorithm characteristics:")
print("  Time complexity: O(n) for n discount tiers")
print("  Space complexity: O(n) when retaining one result per tier")
print("  The procedure evaluates one EOQ-based candidate for each tier.")
print("  Its conclusions rely on the standard EOQ and all-units discount assumptions.")

print("\nBase-example candidates:")
for result in all_results:
    print(
        f"  Tier {result['tier']}: EOQ={result['eoq']:.2f}, "
        f"candidate quantity={result['adjusted_qty']:.2f}, "
        f"annual cost=${result['total_cost']:,.2f}"
    )

print("\nThe selected tier is the one with the minimum evaluated annual cost.")